# PhysX-Omni 论文精读 第四步：Baseline 与实验严谨梳理

论文地址：[https://arxiv.org/abs/2605.21572v1](https://arxiv.org/abs/2605.21572v1)  
代码地址：[https://github.com/physx-omni/PhysX-Omni](https://github.com/physx-omni/PhysX-Omni)  
本地代码：`C:\Users\robot\Documents\成长学习库\physx-omni-assets\code\PhysX-Omni`，当前 `git` 版本 `46fa1cd`  
本地论文源码：`C:\Users\robot\Documents\成长学习库\physx-omni-author-sources\src\main.tex`  
本地官方 demo 复现输出：`C:\Users\robot\physx_outputs\official_demo_full`  
本地 benchmark smoke 输出：`C:\Users\robot\Documents\成长学习库\physx-omni-assets\code\PhysX-Omni\benchmark\tiny_example\generated`

## 0. 这一部分先给结论

PhysX-Omni 的实验不是只在“图像看起来像不像”上做比较，而是分成两套评价：

1. **有 GT 的 conventional evaluation**：在 PhysXVerse 和 PhysX-Mobility 上比较几何与物理属性，指标包括 PSNR、Chamfer Distance、F-score、绝对尺度、材料、affordance、运动学、描述。
2. **无 GT 的 PhysX-Bench**：把生成结果渲染成图像或视频，再由开源 VLM `Qwen/Qwen3.5-122B-A10B` 按统一 prompt 打分，覆盖视觉质量、多视角一致性、尺度、affordance、运动学、材料、描述。

论文里的核心实验结论要分开看：

- 在 **PhysXVerse conventional 指标**上，PhysX-Omni 对所有列都是最优。
- 在 **PhysX-Mobility conventional 指标**上，PhysX-Omni 在 PSNR、CD、绝对尺度、材料、affordance、运动学、描述上最好，但 **F-score 不是最好**，PhysX-Anything 是 `89.51`，PhysX-Omni 是 `88.50`。
- 在 **PhysX-Bench** 上，PhysX-Omni 不是几何视觉指标第一：MonoArt 的 CLIP、3D consistency、visual quality 都更高。PhysX-Omni 的优势主要集中在 simulation-ready 物理维度：尺度、材料、affordance、运动学、描述。
- 最接近的 baseline 是 **PhysX-Anything**。论文认为 PhysX-Anything 的瓶颈在文本 voxel index 表示和显式 segmentation 中间阶段；PhysX-Omni 用 template-based RLE 直接生成高分辨率结构，减少 segmentation 误差传播。
- 我们本地已经跑通的是 **官方生成主流程**和 **benchmark 聚合/分母校验 smoke**。这证明代码路径可执行，但不能把它说成论文 Table 1 / Table 2 的完整复现，因为 full benchmark 还需要各 baseline 输出、condition images、渲染资产、VLM 服务和全量 manifest。

## 1. Baseline 分类

论文比较的 baseline 可以分成三类：

| 方法 | 主要能力 | 论文中可比指标 | 关键局限 |
|---|---:|---|---|
| Articulate-Anything | 生成或构建 articulated asset | geometry、kinematic | 不输出完整尺度、材料、affordance、description，因此很多列为 `--` |
| MonoArt | 单图 articulated 3D/运动估计，强依赖几何生成先验 | geometry、kinematic | 几何强，但不是完整 simulation-ready 物理属性生成器 |
| PhysXGen | 图像到带物理属性的 3D asset | geometry、scale、material、affordance、kinematic、description | 物理属性覆盖较多，但不是 PhysX-Anything/PhysX-Omni 这种统一 sim-ready 表达 |
| PhysX-Anything | 直接前作，文本表示 simulation-ready 资产 | geometry 与物理属性全列 | 依赖显式 segmentation，文本 voxel index 表示较冗长且易损失结构 |
| PhysX-Omni | 本文方法，统一 rigid/deformable/articulated sim-ready 生成 | 全列 | 训练和 full benchmark 复现成本高 |

更详细的 baseline 对照见：

- `C:\Users\robot\Documents\成长学习库\physx_omni_step4_baseline_matrix.md`

## 2. Conventional Evaluation 该怎么读

论文的 conventional evaluation 使用有 GT 的 PhysXVerse 和 PhysX-Mobility。几何部分对生成资产和 GT 资产从 30 个视角渲染，然后平均计算 PSNR、CD、F-score。物理属性部分延续 PhysX-Anything protocol：

- absolute scale：预测尺度与 GT 尺度的 MSE，越低越好。
- material / affordance / description：基于 heatmap 的 PSNR，越高越好。
- kinematic：关节轴位置、方向、关节类型、运动范围等 articulation 参数的 MSE 或对应评分，论文表中越高越好。

关键表格结论：

| 数据集 | 主要观察 |
|---|---|
| PhysXVerse | PhysX-Omni 全列第一。CD 从 MonoArt 的 `7.03` 降到 `2.95`，下降约 `58.0%`；相对 PhysX-Anything 的 `37.06` 下降约 `92.0%`。 |
| PhysXVerse | 绝对尺度误差从 PhysX-Anything 的 `298.19` 降到 `2.79`，下降约 `99.1%`。运动学从 `0.4191` 到 `0.9185`，约 `2.19x`。 |
| PhysX-Mobility | PhysX-Omni 在大多数指标上第一，但 F-score 不是第一。PhysX-Anything `89.51` 高于 PhysX-Omni `88.50`。 |
| PhysX-Mobility | CD 相对 PhysX-Anything 从 `23.13` 降到 `4.70`，下降约 `79.7%`；运动学相对 PhysX-Anything 提升约 `9.6%`。 |

详细数值和逐列解释见：

- `C:\Users\robot\Documents\成长学习库\physx_omni_step4_experiment_tables_analysis.md`

## 3. PhysX-Bench 该怎么读

PhysX-Bench 是论文新增的无 GT 评估框架。它的出发点是：真实 in-the-wild 图片没有 GT 3D 资产，因此不能直接算 CD 或 GT heatmap PSNR。代码里采用的策略是：

1. 把每个方法的输出转成统一证据资产，例如多视角渲染图、mask、affordance heatmap、kinematic video、material simulation video。
2. 为每个指标构造 JSONL manifest。
3. 用开源 VLM `Qwen/Qwen3.5-122B-A10B` 和固定 prompt 输出结构化 JSON 分数。
4. `aggregate_vlm_results.py` 聚合 object-level 和 dataset-level 分数。
5. `validate_denominators.py` 校验每个 method / dataset / metric 的分母，避免漏样本后虚高。

PhysX-Bench 表格的严谨读法：

- MonoArt 在几何三个维度第一：CLIP `0.835`、3D consistency `82.56`、visual quality `96.20`。
- PhysX-Omni 在物理维度第一：scale `64.26`、material `59.89`、affordance `70.57`、kinematic `80.72`、description `39.02`。
- 因此不能简单说 PhysX-Omni 在 PhysX-Bench “所有指标最佳”。准确说法是：**MonoArt 更强于几何外观，PhysX-Omni 更强于 simulation-ready 物理属性与语义一致性，整体更均衡。**

代码层面的 benchmark 映射见：

- `C:\Users\robot\Documents\成长学习库\physx_omni_step4_benchmark_code_mapping.md`

## 4. Baseline 与代码输出格式的对应关系

`benchmark/README.md` 明确要求不同方法的输出放在 `physx_result/<method_result_folder>/<object_id>/` 下。`benchmark/scripts/common.sh` 中的 `source_folder()` 给出默认命名：

| 方法 | 默认结果目录 |
|---|---|
| ours | `ours_<dataset>_181500` |
| physxanything | `output_physxanything_<dataset>` |
| physxgen | `outputs_physxgen_<dataset>` |
| articulateanything | `output_articulateanything_<dataset>` |

这说明 benchmark 不是直接调用各 baseline 模型从零生成，而是假设 baseline 输出已经按约定准备好。然后每个指标从这些输出里取不同证据：

| 指标 | 代码入口 | 主要证据 |
|---|---|---|
| RQS | `benchmark/scripts/run_rqs.sh` | 4 张 rendered views + 质量参考图 |
| MCS | `benchmark/scripts/run_mcs.sh` | 多视角 rendered views |
| DCS | `benchmark/scripts/run_dcs.sh` | 彩色渲染图 + 同视角黑白 part mask + reference description |
| DQS | `benchmark/scripts/run_dqs.sh` | condition image + 生成的 dimension 元数据 |
| APS | `benchmark/scripts/run_aps.sh` | condition image + affordance heatmap views |
| KPS | `benchmark/scripts/run_kps.sh` | condition image + standardized articulation video |
| MPS | `benchmark/scripts/run_mps.sh` | condition image + water/floor simulation videos + material params |

这个代码设计也解释了为什么 full reproduction 比单张 demo 复杂得多：不仅要生成资产，还要为每个 baseline 和每个 metric 生成证据资产。

## 5. 本地复现证据与边界

已完成：

- 官方 demo full path 成功：`C:\Users\robot\physx_outputs\official_demo_full\repro_summary.json`
  - `status=success`
  - `mode=4bit`
  - `detected_parts=7`
  - `total_voxels=22031`
  - `elapsed_sec=393.95`
  - 输出包括 `coord_*.txt`、`ind_*.npy`、`ind_*.ply`、`allind.ply`、`basic_info.json`、`basic.urdf`、`basic.xml`、mesh 目录。
- benchmark tiny smoke 成功：
  - `summary.csv` 中 `ours,mobility,RQS,count=1,mean=100.0`
  - `denominator_validation.csv` 中 `count_mismatch=False`
  - 这验证了 manifest -> fake VLM result -> aggregation -> denominator validation 链路。

未完成，不能夸大：

- 没有全量准备 `physx_result/ours_*`、`output_physxanything_*`、`outputs_physxgen_*`、`output_articulateanything_*`。
- 没有跑全量 RQS/MCS/DCS/DQS/APS/KPS/MPS。
- 没有在本地生成论文 Table 1 / Table 2 的完整复现实验数值。
- 当前 Windows 本地 benchmark smoke 暴露两个平台兼容问题：
  - MSYS bash 会调用 WindowsApps 的 `python3.exe` 占位程序，需映射到真实 `python` 或在 Linux/4090 环境跑。
  - `aggregate_vlm_results.py` 在 Windows 目录扫描时优先用 `rg --files`，但只筛 `/result.json`，会漏掉反斜杠路径；传入具体 `result.json` 文件可绕过。

## 6. 下一步 full benchmark 的质量复现顺序

为了避免“跑了但分母不一致”的问题，后续建议按这个顺序推进：

1. 在 4090 Linux 环境整理 `physx_result/`，确认四类 baseline 输出目录完整。
2. 先跑 `RUN_VLM=0 LIMIT=5` 的各 metric manifest 和资产生成，检查 manifest 中 ready / missing-zero 状态。
3. 跑 RQS/MCS 的 render path，因为它们依赖最少，最适合先测端到端。
4. 跑 KPS，确认 `basic.xml`、`mesh/basic.urdf`、`joint_actor/.../mobility.urdf` 三类方法路径都能渲染成标准视频。
5. 跑 MPS，单独检查 watertight remesh、water/floor simulation 视频是否稳定。
6. 启动 `Qwen/Qwen3.5-122B-A10B` 或兼容 VLM 服务，逐 metric 开始 `RUN_VLM=1`。
7. 每次 metric 完成后立刻跑 `aggregate_vlm_results.py` 和 `validate_denominators.py`，只报告 denominator validation 无 mismatch 的分数。

这一部分的 Jupyter 版本：

- `C:\Users\robot\Documents\成长学习库\physx_omni_paper_baselines_experiments_step4.ipynb`


# PhysX-Omni 第四步附录 A：Baseline 矩阵与可比性分析

## 1. 为什么 baseline 不能只看谁分数高

PhysX-Omni 论文里的 baseline 覆盖了不同问题定义：

- 有些方法主要做 articulated object。
- 有些方法主要做几何生成或重建。
- 有些方法开始输出物理属性。
- PhysX-Anything 和 PhysX-Omni 才更接近“从单图生成 simulation-ready physical 3D asset”的完整目标。

因此表中的 `--` 不是 0 分，而是方法没有对应输出，或不适合按该物理属性指标评估。

## 2. Baseline 逐个拆解

### 2.1 Articulate-Anything

定位：

- 面向 articulated object 的生成或构建。
- 更关注可动部件、关节、运动结构，而不是完整材料、尺度、affordance、description。

论文里怎么评：

- conventional table 中有 PSNR、CD、F-score、kinematic。
- scale、material、affordance、description 为 `--`。
- PhysX-Bench 中也主要参与 geometry 和 kinematic，物理属性列大多为 `--`。

严谨结论：

- 它不是 PhysX-Omni 的完整同类竞品。
- 它的存在主要是为了说明：单纯 articulated generation 不能覆盖完整 simulation-ready 资产。

### 2.2 MonoArt

定位：

- 从单图推断 articulated 3D 和运动结构。
- 论文讨论中指出 MonoArt 借助强几何生成先验，因此在外观/几何类评估上表现强。

论文里怎么评：

- conventional table 中有几何和 kinematic。
- PhysX-Bench 中几何三项最好：CLIP `0.835`、3D consistency `82.56`、visual quality `96.20`。
- 但 scale、material、affordance、description 为 `--`。

严谨结论：

- MonoArt 是几何外观强 baseline。
- 它证明 PhysX-Omni 并不是在所有视觉几何指标上无条件第一。
- 但 MonoArt 不解决完整物理属性建模，因此 simulation-ready 维度不完整。

### 2.3 PhysXGen

定位：

- 从图像生成带部分物理属性的 3D 资产。
- 论文 related work 中把 PhysXGen 放在物理属性生成方向，强调它可以生成尺度、密度等基本物理属性。

论文里怎么评：

- conventional table 中全列可比：geometry、scale、material、affordance、kinematic、description。
- PhysX-Bench 中也参与 geometry、scale、affordance、kinematic、description；material 在表中为 `--`。

严谨结论：

- PhysXGen 比 Articulate-Anything/MonoArt 更接近 PhysX-Omni 的物理属性目标。
- 但它不是 PhysX-Anything/PhysX-Omni 这种更完整的 simulation-ready 统一输出。

### 2.4 PhysX-Anything

定位：

- 最接近 PhysX-Omni 的前作。
- 使用 VLM 和纯文本表示来建模 simulation-ready physical 3D assets。

论文认为的瓶颈：

- 依赖显式 segmentation stage。
- 最终质量受 segmentation module 限制。
- text-based voxel indices 表达冗长，对复杂拓扑、细部结构、邻近部件连续性不友好。

论文里怎么评：

- conventional table 与 PhysX-Omni 全列可比。
- PhysX-Bench 也全列可比。
- 在 PhysX-Mobility 的 F-score 上，PhysX-Anything `89.51` 高于 PhysX-Omni `88.50`，这是必须保留的细节。

严谨结论：

- PhysX-Anything 是本文最重要的直接 baseline。
- PhysX-Omni 的主要 claim 应该理解成：在更紧凑的 template-based RLE 表示和更大数据集下，整体 geometry/physical/kinematic 能力明显提升，而不是每个单项都绝对第一。

## 3. Baseline 输出和 benchmark 代码格式

`benchmark/scripts/common.sh` 里把方法映射到默认输出目录：

| benchmark method | source folder |
|---|---|
| `ours` | `ours_<dataset>_181500` |
| `physxanything` | `output_physxanything_<dataset>` |
| `physxgen` | `outputs_physxgen_<dataset>` |
| `articulateanything` | `output_articulateanything_<dataset>` |

每个 object 目录下面，不同方法需要不同证据：

| 方法 | 常见几何/运动文件 | 物理属性文件 |
|---|---|---|
| ours | `basic.xml`、`basic.urdf`、`sample.glb` 或 mesh outputs | `basic_info.json` 或 `basic_info.txt`、affordance/description masks |
| physxanything | 通常与 ours 相近，`basic.xml`、`basic_info.*` | 同上 |
| physxgen | `texture.glb`、`mesh/basic.urdf`、`scale.npy` | 部分物理属性或单独 metadata |
| articulateanything | `joint_actor/iter_0/seed_0/mobility.urdf` | 通常不包含完整材料/尺度/描述 |

这也解释了 full benchmark 的工作量：它不是只跑一个生成脚本，而是需要把每个 baseline 的输出转成统一证据资产后再评分。

## 4. 对比时最容易犯的错误

1. 把 `--` 当成 0 分。正确理解是“不适用或没有该输出”。
2. 把 PhysX-Bench 的几何项和 conventional geometry 混为一谈。前者是 VLM/CLIP/多视角视觉判断，后者是有 GT 的渲染或几何误差。
3. 只看 PhysX-Omni 的物理维度优势，忽略 MonoArt 在 PhysX-Bench 几何项上更高。
4. 把我们本地官方 demo 的成功说成论文 benchmark 成功。demo 证明主流程可跑，benchmark 需要 baseline 对齐和分母校验。


# PhysX-Omni 第四步附录 B：论文实验表格逐项分析

## 1. Conventional Metrics 表

论文表格列：

- Geometry：PSNR 越高越好，Chamfer Distance 越低越好，F-score 越高越好。
- Physical Attributes：Absolute scale 越低越好；Material、Affordance、Kinematic、Description 越高越好。
- CD 单位是 `x10^-3`。
- F-score 单位是 `x10^-2`，阈值 `0.05`。

### 1.1 PhysXVerse

| Method | PSNR ↑ | CD ↓ | F-score ↑ | Abs. scale ↓ | Material ↑ | Affordance ↑ | Kinematic ↑ | Description ↑ |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| Articulate-Anything | 14.03 | 48.77 | 46.44 | -- | -- | -- | 0.2952 | -- |
| MonoArt | 19.68 | 7.03 | 85.27 | -- | -- | -- | 0.3805 | -- |
| PhysXGen | 19.41 | 15.19 | 83.56 | 309.31 | 16.51 | 9.40 | 0.3494 | 11.84 |
| PhysX-Anything | 15.97 | 37.06 | 40.46 | 298.19 | 15.65 | 10.50 | 0.4191 | 21.38 |
| PhysX-Omni | 21.52 | 2.95 | 91.28 | 2.79 | 27.23 | 21.47 | 0.9185 | 31.05 |

解释：

- PhysX-Omni 在 PhysXVerse 上全列最优。
- 相对 MonoArt，CD 从 `7.03` 到 `2.95`，下降约 `58.0%`。
- 相对 PhysX-Anything，CD 从 `37.06` 到 `2.95`，下降约 `92.0%`。
- 绝对尺度误差相对 PhysX-Anything 从 `298.19` 到 `2.79`，下降约 `99.1%`。
- Kinematic 从 PhysX-Anything 的 `0.4191` 到 `0.9185`，约 `2.19x`。
- 这张表支持论文的主要 claim：template-based RLE 和更统一的 physical representation 对复杂结构、尺度和 articulation 有明显帮助。

### 1.2 PhysX-Mobility

| Method | PSNR ↑ | CD ↓ | F-score ↑ | Abs. scale ↓ | Material ↑ | Affordance ↑ | Kinematic ↑ | Description ↑ |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| Articulate-Anything | 15.02 | 16.09 | 66.95 | -- | -- | -- | 0.6396 | -- |
| MonoArt | 16.46 | 6.35 | 87.41 | -- | -- | -- | 0.4351 | -- |
| PhysXGen | 15.75 | 35.32 | 79.62 | 46.58 | 16.02 | 8.73 | 0.3884 | 11.60 |
| PhysX-Anything | 16.57 | 23.13 | 89.51 | 22.58 | 22.58 | 16.29 | 0.7852 | 26.28 |
| PhysX-Omni | 18.38 | 4.70 | 88.50 | 2.78 | 24.09 | 16.58 | 0.8603 | 28.40 |

解释：

- PhysX-Omni 在 PSNR、CD、absolute scale、material、affordance、kinematic、description 上最好。
- F-score 不是最好：PhysX-Anything `89.51`，PhysX-Omni `88.50`。
- 相对 PhysX-Anything，CD 从 `23.13` 到 `4.70`，下降约 `79.7%`。
- 相对 MonoArt，CD 从 `6.35` 到 `4.70`，下降约 `26.0%`。
- 相对 PhysX-Anything，absolute scale 从 `22.58` 到 `2.78`，下降约 `87.7%`。
- Kinematic 相对 PhysX-Anything 提升约 `9.6%`。

严谨结论：

- PhysX-Mobility 上，PhysX-Omni 不是所有 geometry 子指标全胜。
- 更准确的说法是：PhysX-Omni 在几何误差、尺度、物理属性和 articulation 上整体更好，但 F-score 与 PhysX-Anything 非常接近且略低。

## 2. PhysX-Bench 表

PhysX-Bench 是无 GT 的 VLM benchmark，列包括：

- Geometry：CLIP、3D Consistency、Visual Quality，越高越好。
- Physical Attributes：Absolute scale、Material、Affordance、Kinematic、Description，越高越好。

| Method | CLIP ↑ | 3D Consistency ↑ | Visual Quality ↑ | Abs. scale ↑ | Material ↑ | Affordance ↑ | Kinematic ↑ | Description ↑ |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| Articulate-Anything | 0.554 | 55.27 | 88.46 | -- | -- | -- | 71.25 | -- |
| MonoArt | 0.835 | 82.56 | 96.20 | -- | -- | -- | 68.32 | -- |
| PhysXGen | 0.803 | 73.50 | 85.93 | 24.21 | -- | 66.07 | 69.17 | 22.24 |
| PhysX-Anything | 0.547 | 52.71 | 70.81 | 50.20 | 44.70 | 59.96 | 65.99 | 26.89 |
| PhysX-Omni | 0.767 | 64.48 | 90.00 | 64.26 | 59.89 | 70.57 | 80.72 | 39.02 |

解释：

- MonoArt 的三项几何指标最高。论文也承认 MonoArt 在若干 geometry-related metrics 上更强。
- PhysX-Omni 在所有物理属性列中最高。
- 相对 PhysX-Anything，PhysX-Omni 的 Kinematic 从 `65.99` 到 `80.72`，提升约 `22.3%`。
- 相对 PhysX-Anything，Scale 从 `50.20` 到 `64.26`，提升约 `28.0%`。
- 相对 PhysX-Anything，Material 从 `44.70` 到 `59.89`，提升约 `34.0%`。
- 相对 PhysX-Anything，Description 从 `26.89` 到 `39.02`，提升约 `45.1%`。

严谨结论：

- PhysX-Bench 支持的是“PhysX-Omni 在 simulation-ready 物理属性上显著更强”。
- 它不支持“PhysX-Omni 在所有无 GT 几何视觉指标上最好”。
- 论文把这一点解释为：MonoArt 借助强 TRELLIS 几何 pipeline 得到更高外观指标，但缺少 part-level motion 与物理交互建模。

## 3. Human alignment

论文补充了 PhysX-Bench 自动评分与人工偏好的相关性：

- scale、affordance、material、description：Spearman `rho=1.0`。
- kinematic：Spearman `rho=1.0`，Pearson `r=0.992`。
- geometry：Spearman `rho=0.8`，Pearson `r=0.803`。

这说明 PhysX-Bench 的自动排序与人工偏好高度一致。但这里仍要注意：

- 相关性高说明 benchmark 排序合理，不代表绝对分数没有 VLM 偏差。
- VLM prompt 固定、输入证据固定、分母校验固定，是减少偏差的关键。

## 4. Ablation 的严谨边界

论文 ablation 聚焦 geometry representation：

- baseline：直接用 text-based voxel indices 表示 3D structure，类似 PhysX-Anything 的路径。
- ours：template-based RLE，显式且紧凑地编码高分辨率 3D structure。

论文文字结论：

- 新表示提升 conventional metrics 和 PhysX-Bench。
- 主要改善 kinematic 和 absolute scale。
- qualitative ablation 显示 baseline 在 stroller、tractor 等复杂结构上出现结构歧义、局部几何不完整、articulated components 不一致。

严谨边界：

- 论文没有把所有 ablation 分解成非常细的多因素表，例如单独拆数据集大小、训练 schedule、decoder 等。
- 因此当前能确认的是“表示法 + 简化 pipeline”的组合贡献很强；不能从表格中完全隔离每个工程因素的独立贡献。


# PhysX-Omni 第四步附录 C：Benchmark 代码与论文实验对应

## 1. 代码总流程

本地 benchmark 代码在：

`C:\Users\robot\Documents\成长学习库\physx-omni-assets\code\PhysX-Omni\benchmark`

`benchmark/README.md` 的主流程可以压缩成：

```text
physx_result/ baseline outputs
  -> 生成每个 metric 需要的证据资产
  -> 构造每个 metric 的 manifest
  -> 用固定 prompt 调 VLM 产生 JSON 分数
  -> aggregate_vlm_results.py 聚合
  -> validate_denominators.py 校验分母
```

这个结构和论文 PhysX-Bench 对应：PhysX-Bench 不直接读模型内部参数打分，而是把物理属性转成可视化证据，再让 VLM 按统一 rubric 判断。

## 2. 配置入口

默认配置模板：

`benchmark/configs/paths.example.yaml`

关键字段：

| 字段 | 作用 |
|---|---|
| `physx_result_root` | baseline 输出根目录 |
| `benchmark_asset_root` | 渲染图、视频、heatmap 等证据资产输出根目录 |
| `benchmark_manifest_root` | JSONL/CSV manifest 根目录 |
| `benchmark_result_root` | VLM raw output、object score、summary score 输出根目录 |
| `vlm_model_path` | 默认 `Qwen/Qwen3.5-122B-A10B` |
| `condition_image_root` | DQS/APS/KPS/MPS 共用输入图片 |
| `rendered_view_root` | RQS/MCS/DCS 使用的 rendered views |
| `material_metric_json_root` | MPS 使用的材料参数 JSON |

脚本公共逻辑：

`benchmark/scripts/common.sh`

其中 `METHODS`、`DATASETS`、`RUN_VLM`、`RUN_AGGREGATE`、`RUN_VALIDATE`、`LIMIT` 都在这里统一处理。

## 3. 指标到代码的逐项对应

| PhysX-Bench 指标 | 论文含义 | 脚本 | prompt | 聚合名 |
|---|---|---|---|---|
| RQS | rendered visual quality | `benchmark/scripts/run_rqs.sh` | `benchmark/prompts/prompts_quality.yaml` | `RQS` |
| MCS | multi-view consistency | `benchmark/scripts/run_mcs.sh` | `benchmark/prompts/prompts_consistency.yaml` | `MCS` |
| DCS | description / mask consistency | `benchmark/scripts/run_dcs.sh` | `benchmark/prompts/prompts_description.yaml` | `DCS` |
| DQS | dimension / absolute scale quality | `benchmark/scripts/run_dqs.sh` | `benchmark/prompts/prompts_dimension.yaml` + deterministic scoring | `DQS` |
| APS | affordance plausibility | `benchmark/scripts/run_aps.sh` | `benchmark/prompts/prompts_affordance.yaml` | `APS` |
| KPS | kinematic plausibility | `benchmark/scripts/run_kps.sh` | `benchmark/prompts/prompts_vaps_english.yaml` | `KPS` / `VAPS` |
| MPS | material plausibility | `benchmark/scripts/run_mps.sh` | `benchmark/prompts/prompts_material.yaml` | `MPS` |

## 4. 每个指标的证据资产

### RQS

输入：

- 4 张 rendered views。
- `benchmark/assets/quality_reference/image.png` 作为 1 到 5 的质量参考。

prompt 要求 VLM 输出 1 到 5 分，聚合器会转成 0 到 100 尺度。

### MCS

输入：

- 多视角 rendered views。

prompt 关注：

- global view consistency。
- view-specific failures。
- surface appearance coherence。

它不评价类别是否正确，也不评价 affordance 或部件功能。

### DCS

输入：

- 一张彩色渲染图。
- 同视角黑白 mask。
- reference description。

prompt 中公式：

```text
DCS = round(0.60 * alignment_score + 0.40 * precision_score, 2)
```

这说明 DCS 同时看语义是否对齐和 mask 是否精准。mask 覆盖全物体会被强惩罚。

### DQS

输入：

- condition image。
- 方法输出的 dimension 元数据，例如 `basic_info.json` 里的 `dimension`。

流程：

1. VLM 先从 condition image 估计真实世界最大尺寸。
2. `benchmark/code/scoring/score_dimension_results.py` 读取方法生成的尺寸。
3. 用 symmetric error 计算 DQS。

代码公式：

```text
symmetric_error = 2 * abs(generated_max - vlm_estimated_max) / (generated_max + vlm_estimated_max)
if symmetric_error >= 0.8:
    DQS = 0
else:
    DQS = 100 * (1 - symmetric_error / 0.8)
```

### APS

输入：

- condition image。
- 多视角 affordance heatmap。

prompt 关注：

- relative ranking plausibility。
- salient misranking。
- overall common-sense plausibility。
- 危险区域不应被高亮为强 affordance。

这说明 APS 不是看 heatmap 是否漂亮，而是看交互强弱排序是否符合常识。

### KPS

输入：

- condition image。
- 标准化 articulation video。
- 图像先验 JSON。

代码统一使用：

`benchmark/code/render/kinematic/kinematic_articulation_demo.py`

不同方法的输入：

| 方法 | KPS 渲染文件 |
|---|---|
| ours / physxanything | `basic.xml` |
| physxgen | `mesh/basic.urdf` |
| articulateanything | `joint_actor/iter_0/seed_0/mobility.urdf` |

prompt 中最终分数是 VAPS：

```text
base weights:
prior channel = 0.70
reveal channel = 0.20
global channel = 0.10
```

如果 prior 或 reveal channel inactive，则对应权重不参与归一化。

### MPS

输入：

- condition image。
- water simulation video。
- floor simulation video。
- 生成的 material parameters。

prompt 让 VLM 分别评价：

- Young's modulus。
- Poisson's ratio。
- Density。

公式：

```text
weighted_score = 0.4 * S_E + 0.2 * S_nu + 0.4 * S_rho
MPS = 25 * (weighted_score - 1)
```

所以 MPS 不是 VLM 随意给 0 到 100，而是先给三个 1 到 5 的子分，再按公式转成 0 到 100。

## 5. Aggregation 与 denominator validation

聚合脚本：

`benchmark/code/aggregation/aggregate_vlm_results.py`

它把 raw `result.json` 解析为：

- object-level long rows。
- dataset-level summary。
- submetric summary。

分母校验脚本：

`benchmark/code/validation/validate_denominators.py`

它检查每个 method / dataset / metric：

- manifest expected count。
- ready count。
- missing-zero count。
- raw result count。
- dedup 后 count。
- object score count。
- summary count。
- 是否 count mismatch。

这一步非常关键。没有 denominator validation，就可能出现某个方法因为少跑了难样本而均值虚高。

## 6. 本地 smoke 验证

我在 Windows 本地用 PowerShell 版 smoke 跑通了官方代码模块：

```text
build_render_view_manifest.py
  -> fake result.json
  -> aggregate_vlm_results.py
  -> validate_denominators.py
```

输出：

`C:\Users\robot\Documents\成长学习库\physx-omni-assets\code\PhysX-Omni\benchmark\tiny_example\generated\benchmark_results\summary.csv`

内容要点：

```text
method,dataset,metric,count,mean,std
ours,mobility,RQS,1,100.0,0.0
```

分母校验：

`C:\Users\robot\Documents\成长学习库\physx-omni-assets\code\PhysX-Omni\benchmark\tiny_example\generated\benchmark_results\denominator_validation.csv`

内容要点：

```text
method,dataset,metric,expected_count,...,summary_count,summary_mean,...,count_mismatch
ours,mobility,RQS,1,...,1,100.0,...,False
```

边界：

- 这是 benchmark 辅助链路 smoke，不是 full PhysX-Bench。
- 没有使用真实 VLM 输出，fake result 只用于验证聚合和分母校验。
- 在 Windows 下发现 `aggregate_vlm_results.py` 扫目录时会因为 `rg --files` 输出反斜杠路径而漏掉 `result.json`。传入具体 `result.json` 文件可以绕过；在 4090 Linux 环境预计不会触发这个 Windows 路径问题。

## 7. 后续 full benchmark 命令骨架

先检查资产和 manifest，不跑 VLM：

```bash
CONFIG=benchmark/configs/paths.yaml \
METHODS="ours physxanything physxgen articulateanything" \
DATASETS="mobility verse inthewild" \
RUN_VLM=0 \
LIMIT=5 \
bash benchmark/scripts/run_rqs.sh
```

RQS/MCS 先跑，因为依赖最少：

```bash
RUN_RENDER=1 bash benchmark/scripts/run_rqs.sh
RUN_RENDER=1 bash benchmark/scripts/run_mcs.sh
```

KPS 需要标准 articulation video：

```bash
RENDER_KPS=1 bash benchmark/scripts/run_kps.sh
```

MPS 需要 watertight proxy 和 water/floor simulation：

```bash
RUN_WATERTIGHT=1 RENDER_MATERIAL=1 bash benchmark/scripts/run_mps.sh
```

每个 metric 结束后都跑：

```bash
python3 benchmark/code/aggregation/aggregate_vlm_results.py \
  --results-root benchmark/benchmark_results/raw_vlm_outputs \
  --object-csv benchmark/benchmark_results/object_level_scores/object_scores_long.csv \
  --summary-csv benchmark/benchmark_results/dataset_level_scores/dataset_metric_summary.csv \
  --submetric-csv benchmark/benchmark_results/dataset_level_scores/dataset_submetric_summary.csv \
  --errors-jsonl benchmark/benchmark_results/logs/aggregate_errors.jsonl

python3 benchmark/code/validation/validate_denominators.py \
  --manifest-root benchmark/benchmark_manifests \
  --results-root benchmark/benchmark_results/raw_vlm_outputs \
  --object-csv benchmark/benchmark_results/object_level_scores/object_scores_long.csv \
  --summary-csv benchmark/benchmark_results/dataset_level_scores/dataset_metric_summary.csv \
  --output-csv benchmark/benchmark_results/logs/denominator_validation.csv \
  --errors-jsonl benchmark/benchmark_results/logs/denominator_validation_errors.jsonl \
  --strict
```


# Notebook calculations and local evidence checks

The cells below re-enter the paper tables as dataframes, compute the main relative changes, and read the local smoke/demo evidence generated during reproduction.


In [1]:
from pathlib import Path
import json
import pandas as pd

workspace = Path.cwd()
repo = workspace / "physx-omni-assets" / "code" / "PhysX-Omni"
official_demo = Path.home() / "physx_outputs" / "official_demo_full"
smoke = repo / "benchmark" / "tiny_example" / "generated"

print("workspace", workspace)
print("repo exists", repo.exists())
print("official demo exists", official_demo.exists())
print("benchmark smoke exists", smoke.exists())


workspace C:\Users\robot\Documents\成长学习库
repo exists True
official demo exists True
benchmark smoke exists True


In [2]:
verse = pd.DataFrame([
    ["Articulate-Anything", 14.03, 48.77, 46.44, None, None, None, 0.2952, None],
    ["MonoArt", 19.68, 7.03, 85.27, None, None, None, 0.3805, None],
    ["PhysXGen", 19.41, 15.19, 83.56, 309.31, 16.51, 9.40, 0.3494, 11.84],
    ["PhysX-Anything", 15.97, 37.06, 40.46, 298.19, 15.65, 10.50, 0.4191, 21.38],
    ["PhysX-Omni", 21.52, 2.95, 91.28, 2.79, 27.23, 21.47, 0.9185, 31.05],
], columns=["Method", "PSNR", "CD", "F-score", "AbsScale", "Material", "Affordance", "Kinematic", "Description"])

mobility = pd.DataFrame([
    ["Articulate-Anything", 15.02, 16.09, 66.95, None, None, None, 0.6396, None],
    ["MonoArt", 16.46, 6.35, 87.41, None, None, None, 0.4351, None],
    ["PhysXGen", 15.75, 35.32, 79.62, 46.58, 16.02, 8.73, 0.3884, 11.60],
    ["PhysX-Anything", 16.57, 23.13, 89.51, 22.58, 22.58, 16.29, 0.7852, 26.28],
    ["PhysX-Omni", 18.38, 4.70, 88.50, 2.78, 24.09, 16.58, 0.8603, 28.40],
], columns=verse.columns)

display(verse)
display(mobility)


,Method,PSNR,CD,F-score,AbsScale,Material,Affordance,Kinematic,Description
0,Articulate-Anything,14.03,48.77,46.44,NaN,NaN,NaN,0.2952,NaN
1,MonoArt,19.68,7.03,85.27,NaN,NaN,NaN,0.3805,NaN
2,PhysXGen,19.41,15.19,83.56,309.31,16.51,9.40,0.3494,11.84
3,PhysX-Anything,15.97,37.06,40.46,298.19,15.65,10.50,0.4191,21.38
4,PhysX-Omni,21.52,2.95,91.28,2.79,27.23,21.47,0.9185,31.05


,Method,PSNR,CD,F-score,AbsScale,Material,Affordance,Kinematic,Description
0,Articulate-Anything,15.02,16.09,66.95,NaN,NaN,NaN,0.6396,NaN
1,MonoArt,16.46,6.35,87.41,NaN,NaN,NaN,0.4351,NaN
2,PhysXGen,15.75,35.32,79.62,46.58,16.02,8.73,0.3884,11.60
3,PhysX-Anything,16.57,23.13,89.51,22.58,22.58,16.29,0.7852,26.28
4,PhysX-Omni,18.38,4.70,88.50,2.78,24.09,16.58,0.8603,28.40


In [3]:
def lower_is_better_reduction(old, new):
    return 100.0 * (old - new) / old

def higher_is_better_gain(old, new):
    return 100.0 * (new - old) / old

summary = pd.DataFrame([
    ["Verse CD vs MonoArt", lower_is_better_reduction(7.03, 2.95), "lower is better"],
    ["Verse CD vs PhysX-Anything", lower_is_better_reduction(37.06, 2.95), "lower is better"],
    ["Verse AbsScale vs PhysX-Anything", lower_is_better_reduction(298.19, 2.79), "lower is better"],
    ["Verse Kinematic vs PhysX-Anything", higher_is_better_gain(0.4191, 0.9185), "higher is better"],
    ["Mobility CD vs PhysX-Anything", lower_is_better_reduction(23.13, 4.70), "lower is better"],
    ["Mobility CD vs MonoArt", lower_is_better_reduction(6.35, 4.70), "lower is better"],
    ["Mobility AbsScale vs PhysX-Anything", lower_is_better_reduction(22.58, 2.78), "lower is better"],
    ["Mobility Kinematic vs PhysX-Anything", higher_is_better_gain(0.7852, 0.8603), "higher is better"],
], columns=["Comparison", "Relative change percent", "Direction"])
summary["Relative change percent"] = summary["Relative change percent"].round(2)
display(summary)
print("Mobility F-score caveat: PhysX-Anything 89.51 > PhysX-Omni 88.50")


,Comparison,Relative change percent,Direction
0,Verse CD vs MonoArt,58.04,lower is better
1,Verse CD vs PhysX-Anything,92.04,lower is better
2,Verse AbsScale vs PhysX-Anything,99.06,lower is better
3,Verse Kinematic vs PhysX-Anything,119.16,higher is better
4,Mobility CD vs PhysX-Anything,79.68,lower is better
5,Mobility CD vs MonoArt,25.98,lower is better
6,Mobility AbsScale vs PhysX-Anything,87.69,lower is better
7,Mobility Kinematic vs PhysX-Anything,9.56,higher is better


Mobility F-score caveat: PhysX-Anything 89.51 > PhysX-Omni 88.50


In [4]:
bench = pd.DataFrame([
    ["Articulate-Anything", 0.554, 55.27, 88.46, None, None, None, 71.25, None],
    ["MonoArt", 0.835, 82.56, 96.20, None, None, None, 68.32, None],
    ["PhysXGen", 0.803, 73.50, 85.93, 24.21, None, 66.07, 69.17, 22.24],
    ["PhysX-Anything", 0.547, 52.71, 70.81, 50.20, 44.70, 59.96, 65.99, 26.89],
    ["PhysX-Omni", 0.767, 64.48, 90.00, 64.26, 59.89, 70.57, 80.72, 39.02],
], columns=["Method", "CLIP", "3DConsistency", "VisualQuality", "AbsScale", "Material", "Affordance", "Kinematic", "Description"])

display(bench)
physxanything = bench[bench.Method == "PhysX-Anything"].iloc[0]
ours = bench[bench.Method == "PhysX-Omni"].iloc[0]
bench_gain = pd.DataFrame([
    ["Scale", higher_is_better_gain(physxanything.AbsScale, ours.AbsScale)],
    ["Material", higher_is_better_gain(physxanything.Material, ours.Material)],
    ["Affordance", higher_is_better_gain(physxanything.Affordance, ours.Affordance)],
    ["Kinematic", higher_is_better_gain(physxanything.Kinematic, ours.Kinematic)],
    ["Description", higher_is_better_gain(physxanything.Description, ours.Description)],
], columns=["Metric", "Ours vs PhysX-Anything gain percent"])
bench_gain["Ours vs PhysX-Anything gain percent"] = bench_gain["Ours vs PhysX-Anything gain percent"].round(2)
display(bench_gain)
print("PhysX-Bench geometry caveat: MonoArt is best on CLIP, 3DConsistency, and VisualQuality.")


,Method,CLIP,3DConsistency,VisualQuality,AbsScale,Material,Affordance,Kinematic,Description
0,Articulate-Anything,0.554,55.27,88.46,NaN,NaN,NaN,71.25,NaN
1,MonoArt,0.835,82.56,96.20,NaN,NaN,NaN,68.32,NaN
2,PhysXGen,0.803,73.50,85.93,24.21,NaN,66.07,69.17,22.24
3,PhysX-Anything,0.547,52.71,70.81,50.20,44.70,59.96,65.99,26.89
4,PhysX-Omni,0.767,64.48,90.00,64.26,59.89,70.57,80.72,39.02


,Metric,Ours vs PhysX-Anything gain percent
0,Scale,28.01
1,Material,33.98
2,Affordance,17.70
3,Kinematic,22.32
4,Description,45.11


PhysX-Bench geometry caveat: MonoArt is best on CLIP, 3DConsistency, and VisualQuality.


In [5]:
summary_csv = smoke / "benchmark_results" / "summary.csv"
validation_csv = smoke / "benchmark_results" / "denominator_validation.csv"

if summary_csv.exists():
    display(pd.read_csv(summary_csv))
else:
    print("missing", summary_csv)

if validation_csv.exists():
    display(pd.read_csv(validation_csv))
else:
    print("missing", validation_csv)


,method,dataset,metric,count,mean,std
0,ours,mobility,RQS,1,100.0,0.0


,method,dataset,metric,expected_count,manifest_ready_count,manifest_missing_zero_count,raw_metric_rows_before_dedup,raw_dedup_count,raw_auto_zero_count,object_score_count,summary_count,summary_mean,duplicate_object_result_keys,missing_raw_after_dedup,extra_raw_after_dedup,missing_object_ids_sample,extra_raw_object_ids_sample,count_mismatch
0,ours,mobility,RQS,1,1,0,1,1,0,1,1,100.0,0,0,0,NaN,NaN,False


In [6]:
summary_path = official_demo / "repro_summary.json"
if summary_path.exists():
    data = json.loads(summary_path.read_text(encoding="utf-8"))
    selected = {k: data.get(k) for k in ["status", "mode", "detected_parts", "parts_to_run", "elapsed_sec", "total_voxels"]}
    print(json.dumps(selected, ensure_ascii=False, indent=2))
    parts = pd.DataFrame(data.get("parts", []))[["part", "coord_text_chars", "voxel_count"]]
    display(parts)
else:
    print("missing", summary_path)


{
  "status": "success",
  "mode": "4bit",
  "detected_parts": 7,
  "parts_to_run": 7,
  "elapsed_sec": 393.95,
  "total_voxels": 22031
}


,part,coord_text_chars,voxel_count
0,0,758,56
1,1,8446,14065
2,2,6771,7570
3,3,749,46
4,4,1416,186
5,5,765,60
6,6,735,48
